# 01 — Prompt template check and prompt lengths (step 1.3)

Builds real prompts with `src/prompts.py` and measures their length in tokens for every split.
These numbers set `max_model_len` for vLLM (Phase 1.5) and `max_seq_length` for training (Phase 3).

Runtime: CPU. BIRD train is unpacked from the HF dataset repo, which takes several minutes.

In [ ]:
import os
REPO_DIR = "/content/text2sql-small-llm"
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/sameerahmed-cloud/text2sql-small-llm.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull
%cd {REPO_DIR}

In [ ]:
from google.colab import drive, userdata
drive.mount("/content/drive")
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")   # BIRD train is in a private HF repo

import json
from src.get_data import unpack_all
from src.data_prep import DRIVE_PROCESSED, db_path

paths = unpack_all(include_bird_train=True)

def read_jsonl(path):
    return [json.loads(line) for line in open(path, encoding="utf-8")]

splits = {n: read_jsonl(DRIVE_PROCESSED / f"{n}.jsonl") for n in ["train", "val", "test_bird", "test_spider"]}
print({n: len(r) for n, r in splits.items()})   # expect 14601, 1634, 1534, 1034

In [ ]:
from transformers import AutoTokenizer
from src.prompts import build_prompt, measure_prompt_lengths

tok = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-Coder-1.5B-Instruct")

# One real Spider prompt, printed in full, as a visual check
r = splits["test_spider"][0]
print(build_prompt(db_path(r, paths), r["question"], r["evidence"], tok))
print("GOLD:", r["gold_sql"])

In [ ]:
# Token lengths for every split (takes a few minutes: 18,803 prompts)
stats = [measure_prompt_lengths(recs, paths, tok, name) for name, recs in splits.items()]